In [8]:
import pandas as pd
from ultralytics import YOLO
import cv_folds


In [9]:
all_metrics = []
n_splits = 5
folds_yaml = cv_folds.create_cv_folds(data_yaml='detection_dataset/yolo-test.yaml', n_splits = n_splits)

In [10]:
for fold, fold_yaml in enumerate(folds_yaml):
    print(f"\n{'='*50}")
    print(f"Training Fold {fold + 1}/5")
    print(f"{'='*50}")
    model = YOLO('weights/yolo11s-finetuned.pt')
    model.train(data=fold_yaml,
                        device=0,
                        batch=16,
                        epochs=50,
                        project='cv_results',
                        name=f'fold_{fold}',
                        exist_ok=True)
    metrics = model.val(
                    data=f'data_{fold}.yaml',
                    split='val',
                    conf=0.5,
                    iou=0.5,
                    plots=True
                )
    fold_metrics = {
            'fold': fold,
            'map50': metrics.box.map50,
            'map': metrics.box.map,
            'precision': metrics.box.mean_results()[0],
            'recall': metrics.box.mean_results()[1],
        }
    all_metrics.append(fold_metrics)


Training Fold 1/5
New https://pypi.org/project/ultralytics/8.4.53 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.13.9 torch-2.12.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=weights/yolo11s-finetuned.pt, momentum=0.937

In [11]:
metrics_df = pd.DataFrame(all_metrics)
metrics_df

,fold,map50,map,precision,recall
0,0,0.834510,0.557759,0.934366,0.849213
1,1,0.703350,0.448154,0.941414,0.715822
2,2,0.882239,0.572496,0.934198,0.898396
3,3,0.878172,0.601378,0.946898,0.896771
4,4,0.740870,0.465438,0.915902,0.750627


In [15]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Функция расчёта среднего и ДИ через t-распределение
def mean_ci(values, conf=0.95):
    mean = np.mean(values)
    se = np.std(values, ddof=1) / np.sqrt(len(values))
    t = stats.t.ppf((1 + conf)/2, len(values)-1)
    margin = t * se
    return mean, margin

# Исходные названия колонок и новые подписи
cols = ['map50', 'map']
labels = ['mAP@0.5', 'mAP@0.5:0.95']

means = []
errors = []
for col in cols:
    m_mean, m_margin = mean_ci(metrics_df[col])
    means.append(m_mean)
    errors.append(m_margin)

# Построение графика
plt.figure(figsize=(8, 6))
bars = plt.bar(labels, means, yerr=errors, capsize=8,
               color=['#3498db', '#2ecc71'],
               edgecolor='black', linewidth=1.2)

print(means, errors)
# Подписи значений
for bar, mean_val, err_val in zip(bars, means, errors):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + err_val + 0.005,
             f'{mean_val:.3f}±{err_val:.3f}', ha='center', va='bottom', fontsize=10)

# Добавляем точки каждого фолда
for i, col in enumerate(cols):
    y_vals = metrics_df[col]
    plt.scatter([labels[i]] * len(y_vals), y_vals,
                color='red', zorder=5, alpha=0.7, s=50)

plt.ylabel('Значение метрики', fontsize=12)
plt.title('YOLO: средние метрики и 95% доверительные интервалы (n=5 фолдов)', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.6)

# Автоматическая подстройка вертикальных границ (можно оставить)
plt.tight_layout()
plt.show()

[np.float64(0.8078282028055346), np.float64(0.5290449110322539)] [np.float64(0.10124769586827664), np.float64(0.08451930546298173)]


<Figure size 800x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>